# Najafi et al. (2020) on DANDI:000016, notebook 4: angles between decoders trained at different times

**Paper.** Najafi F, Elsayed GF, Cao R, Pnevmatikakis E, Latham PE, Cunningham JP, Churchland AK (2020).
Excitatory and inhibitory subnetworks are equally selective during decision-making and emerge
simultaneously during learning. *Neuron* 105, 165-179. [doi:10.1016/j.neuron.2019.09.045](https://doi.org/10.1016/j.neuron.2019.09.045)

**Dataset.** [DANDI:000016](https://dandiarchive.org/dandiset/000016/0.260915.2309), version
0.260915.2309 (doi:10.48324/dandi.000016/0.260915.2309), streamed from the archive. This is the only
published version (2026-09-15); the paper cites no DANDI version (it names the NWB deposit at the
CSHL repository, doi:10.14224/1.37693, from which the dandiset is derived), and the draft holds
the same 135 files.

**Claims.** `claims/000016/10.1016-j.neuron.2019.09.045.yaml`, written from the accepted author
manuscript in PubMed Central (PMC6952547) with its supplement, and the paper's code
(github.com/farznaj/imaging_decisionMaking_exc_inh at commit b7352acd). The claims file decides what
is tested and how; this notebook does not change a claim.

Claims tested here (the full table is in notebook 1):

| Claim | Finding | Notebook | Status |
|---|---|---|---|
| `figs7c-angle-stability` | Similar angle-based stability | 04 | tested |

Inhibitory and subsampled excitatory decoders are trained at every bin (C chosen per bin), and the
angle between the repetition-averaged weight vectors of each pair of bins is computed. For run
time, one random excitatory subsample per session is used where the claims file has 50; a
deviation, recorded.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import najafi as nj
import sessions as S

pd.set_option("display.width", 200)
MICE = [1, 2, 3, 4]
A = nj.asset_urls()
print(f"DANDI:{nj.DANDISET_ID} version {nj.DANDISET_VERSION}: {len(A)} NWB files")
items = [(i, p, u) for i, (p, u) in enumerate(zip(A.path, A.url))]

R = nj.map_sessions(S.decode_angles, items, workers=4)
OK = [r for r in R if r["ok"]]
print(len(R), "sessions loaded,", len(OK), "used; excitatory subsamples per session:", S.N_EXC_ANGLE)

## Data check

In [ ]:
d = pd.DataFrame([dict(mouse=r["mouse"], ok=r["ok"], bins=len(r["bin_times"]), inh=r["n_inh"]) for r in R])
print(d.groupby("mouse").agg(sessions=("ok", "size"), used=("ok", "sum"), bins_min=("bins", "min"), bins_max=("bins", "max"),
                             inh_mean=("inh", "mean")).round(1).to_string())
a = np.concatenate([r["angles_inh"][np.isfinite(r["angles_inh"])] for r in OK] + [r["angles_exc"][np.isfinite(r["angles_exc"])] for r in OK])
print("Angles over all sessions and bin pairs: %.1f to %.1f degrees" % (a.min(), a.max()))

## figs7c-angle-stability

> The angle between decoders trained at different times shows a similar time course of stability
> for excitatory and inhibitory populations.

Criterion: readings (a) the mean angle over all pairs of bins and (b) the mean angle between the
decoder at 0-97 ms before the choice and each other bin, per session and type; unpaired two-tailed
t-test across sessions per mouse at p < 0.05, a claim of absence. Reproduced when no mouse differs.
Bins are those all of a mouse's sessions share.

In [ ]:
rows = []
for m in MICE:
    rr = [r for r in OK if r["mouse"] == m]
    lo = min(r["i1"] + 1 for r in rr); hi = min(len(r["bin_times"]) - (r["i1"] + 1) for r in rr)
    off = np.arange(-lo, hi)
    Ai = np.stack([r["angles_inh"][np.ix_(r["i1"] + 1 + off, r["i1"] + 1 + off)] for r in rr])
    Ae = np.stack([r["angles_exc"][np.ix_(r["i1"] + 1 + off, r["i1"] + 1 + off)] for r in rr])
    j = int(np.where(off == -1)[0][0])
    va_i, va_e = np.nanmean(Ai, axis=(1, 2)), np.nanmean(Ae, axis=(1, 2))
    vb_i, vb_e = np.nanmean(Ai[:, j, :], 1), np.nanmean(Ae[:, j, :], 1)
    rows.append(dict(mouse=m, sessions=len(rr), bins=len(off), a_inh=va_i.mean(), a_exc=va_e.mean(), p_a=nj.ttest(va_i, va_e)[1],
                     b_inh=vb_i.mean(), b_exc=vb_e.mean(), p_b=nj.ttest(vb_i, vb_e)[1]))
    print(f"mouse {m}: angle to the decoder at -1, per bin from offset {off[0]}: inh", np.round(np.nanmean(Ai[:, j, :], 0), 1).tolist())
    print(f"          exc", np.round(np.nanmean(Ae[:, j, :], 0), 1).tolist())
T = pd.DataFrame(rows)
print(T.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("Mice differing, reading (a):", T.mouse[T.p_a < 0.05].tolist(), "; reading (b):", T.mouse[T.p_b < 0.05].tolist())

**Result: (to be written)**

## Summary

(to be written)